# Optimization Tricks in Deep Learning

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 04.07 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/08_optimization_in_deep_learning.ipynb)

---

## 🎯 Learning Objective

Learn the practical optimization tricks — gradient clipping, gradient accumulation, mixed precision — that make training large models feasible.

---

## 📐 Theory

Everything so far in this module has been about *which direction and how far* to step. This
notebook covers three practical realities that determine whether a training run survives long
enough to converge at all, once you leave toy problems for real hardware and real data.

### Gradient clipping

A single unusual batch — an outlier, a numerically unstable loss spike, an unlucky sample near
a discontinuity — can produce a gradient with enormous norm. Combined with a learning rate
tuned for *typical* gradients, one such batch can push weights far outside any region where the
loss behaves sensibly, and training may never recover (recall the exploding-gradient mechanism
from `02.10`, now triggered by data rather than depth). **Gradient clipping by norm** caps the
*overall* gradient vector's length before the update:

$$\mathbf{g} \leftarrow \mathbf{g} \cdot \min\!\left(1, \frac{\tau}{\|\mathbf{g}\|}\right)$$

for a threshold $\tau$. If $\|\mathbf{g}\|\le\tau$ already, nothing changes; if
$\|\mathbf{g}\|>\tau$, the vector is rescaled down to exactly length $\tau$ while keeping its
*direction* — clipping doesn't change which way you step, only how far, which matters because
the direction is still a locally valid descent direction even when its magnitude is untrustworthy.

### Gradient accumulation

GPU memory limits how large a batch you can fit at once. **Gradient accumulation** simulates a
larger effective batch by running several smaller "micro-batches," summing (or averaging) their
gradients *without* resetting between them, and only calling the optimizer step once every $k$
micro-batches:

$$\mathbf{g}_{\text{effective}} = \frac{1}{k}\sum_{i=1}^{k} \mathbf{g}_i$$

If each micro-batch loss is itself an average over its own examples, averaging the $k$
micro-batch gradients reproduces *exactly* the gradient you would have gotten from one large
batch containing all $k$ micro-batches' examples together (provided the micro-batches are equal
size) — accumulation trades memory for a few extra forward/backward passes, not for accuracy.

### Mixed precision training

Modern GPUs compute much faster in float16/bfloat16 than float32 (previewed in `06.01`, fully
covered in `06.06`), but naive float16 training risks numerical underflow: gradients that would
be representable in float32 can round to exactly zero in float16's narrower range. **Mixed
precision training** keeps a float32 "master copy" of the weights, computes the forward/backward
pass in float16 for speed, and (in the classic recipe) *scales the loss up* by a large constant
before backpropagating so small gradients don't underflow, then unscales the resulting gradients
before the optimizer step — recovering the precision-sensitive parts of training while keeping
the fast, cheap arithmetic everywhere else.

### Why these are "engineering," not new math

None of this changes *what* is being optimized — clipping still points along $-\nabla L$
(direction unchanged, magnitude capped), accumulation still computes the exact mini-batch
gradient (just via smaller pieces), mixed precision still computes the same forward/backward
formulas (just in a different number format with compensating scale factors). They're solutions
to *engineering* constraints — unstable batches, limited memory, limited numeric range — layered
on top of the pure mathematics from `04.01`-`04.07`, which is exactly why they combine so
easily with any optimizer covered so far.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A single corrupted training batch spikes the loss for an unclipped run and takes many steps to
recover from; the same run with gradient clipping barely notices the same corrupted batch.

In [ ]:
rng = np.random.default_rng(0)
torch.manual_seed(0)
N = 300
X = torch.tensor(rng.uniform(-2, 2, size=(N, 5)), dtype=torch.float32)
true_w = torch.tensor(rng.normal(size=5), dtype=torch.float32)
y = (X @ true_w + 0.1 * torch.randn(N)).unsqueeze(1)

def make_model():
    torch.manual_seed(1)
    return torch.nn.Sequential(torch.nn.Linear(5, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))

def train(clip_norm, steps=150, batch_size=10, lr=0.1, seed=2, outlier_step=30):
    model = make_model()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    losses = []
    for t in range(steps):
        idx = torch.randint(0, N, (batch_size,), generator=g)
        xb, yb = X[idx].clone(), y[idx].clone()
        if t == outlier_step:
            yb = yb + 200.0  # simulate one corrupted/outlier batch, a single transient event
        opt.zero_grad()
        loss = torch.nn.functional.mse_loss(model(xb), yb)
        loss.backward()
        if clip_norm:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        opt.step()
        with torch.no_grad():
            losses.append(torch.nn.functional.mse_loss(model(X), y).item())
    return losses

losses_noclip = train(clip_norm=None)
losses_clip = train(clip_norm=1.0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy([min(l, 1e5) for l in losses_noclip], color="#C44E52", label="no clipping")
ax.semilogy([min(l, 1e5) for l in losses_clip], color="#4C72B0", label="clip_grad_norm=1.0")
ax.axvline(30, color="gray", ls="--", lw=1, label="corrupted batch hits here")
ax.set_xlabel("training step"); ax.set_ylabel("full-dataset loss (log scale)")
ax.set_title("One bad batch derails unclipped training; clipping absorbs it")
ax.legend()
plt.show()

print(f"Peak loss after the outlier -- no clip: {max(losses_noclip):.2f}, clipped: {max(losses_clip):.2f}")
print(f"Final loss -- no clip: {losses_noclip[-1]:.4f}, clipped: {losses_clip[-1]:.4f}")

## 🐍 Implementation from Scratch

We implement gradient-norm clipping by hand and confirm it matches `torch.nn.utils.clip_grad_norm_`
exactly, then verify that gradient accumulation over several micro-batches reproduces the exact
gradient of one large batch.

In [ ]:
def clip_grad_norm_scratch(parameters, max_norm):
    # model.parameters() returns a GENERATOR -- materialize it to a list first, or the norm
    # computation below would exhaust it, leaving nothing for the mutation loop afterward.
    parameters = list(parameters)
    # Compute the TOTAL norm across all parameters jointly (not per-parameter), matching
    # torch's convention: treat every gradient as one big concatenated vector.
    total_norm_sq = sum((p.grad ** 2).sum().item() for p in parameters if p.grad is not None)
    total_norm = total_norm_sq ** 0.5
    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1:
        for p in parameters:
            if p.grad is not None:
                p.grad.mul_(clip_coef)
    return total_norm

# Verify against torch's built-in on a model with a deliberately huge gradient
torch.manual_seed(3)
model_a = torch.nn.Linear(4, 1)
model_b = torch.nn.Linear(4, 1)
model_b.load_state_dict(model_a.state_dict())  # identical starting weights

x_huge = torch.randn(4) * 100  # large input -> large gradient
for model in (model_a, model_b):
    model.zero_grad()
    loss = (model(x_huge) ** 2).sum()
    loss.backward()

norm_scratch = clip_grad_norm_scratch(model_a.parameters(), max_norm=1.0)
norm_torch = torch.nn.utils.clip_grad_norm_(model_b.parameters(), max_norm=1.0)
print(f"Pre-clip gradient norm: {norm_scratch:.4f} (scratch) vs {norm_torch.item():.4f} (torch)")
grads_match = all(torch.allclose(pa.grad, pb.grad, atol=1e-5) for pa, pb in zip(model_a.parameters(), model_b.parameters()))
print(f"Post-clip gradients match exactly: {grads_match}")

# --- Gradient accumulation: k micro-batches should reproduce one big batch's gradient exactly ---
rng2 = np.random.default_rng(5)
N2 = 64
X2 = torch.tensor(rng2.uniform(-2, 2, size=(N2, 5)), dtype=torch.float32)
w2 = torch.tensor(rng2.normal(size=5), dtype=torch.float32)
y2 = (X2 @ w2 + 0.1 * torch.randn(N2)).unsqueeze(1)

torch.manual_seed(1)
model_big = torch.nn.Sequential(torch.nn.Linear(5, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
torch.manual_seed(1)
model_accum = torch.nn.Sequential(torch.nn.Linear(5, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))

model_big.zero_grad()
torch.nn.functional.mse_loss(model_big(X2), y2).backward()

model_accum.zero_grad()
micro_bs, n_micro = 16, N2 // 16
for i in range(n_micro):
    xb, yb = X2[i*micro_bs:(i+1)*micro_bs], y2[i*micro_bs:(i+1)*micro_bs]
    # Divide by n_micro so summed-then-accumulated gradients average correctly across
    # equal-sized micro-batches -- this is the "loss / accumulation_steps" convention
    (torch.nn.functional.mse_loss(model_accum(xb), yb) / n_micro).backward()

max_diff = max((pb.grad - pa.grad).abs().max().item() for pa, pb in zip(model_big.parameters(), model_accum.parameters()))
print(f"\nOne big batch (size {N2}) vs {n_micro} accumulated micro-batches (size {micro_bs} each):")
print(f"max gradient difference across all parameters: {max_diff:.2e} (floating-point-level agreement)")

## 🤖 Why This Matters for AI

Every large language model's public training recipe mentions gradient clipping (almost always
by global norm, typically $\tau=1.0$) — it's cheap insurance against the rare batch that would
otherwise derail days of training. Gradient accumulation is how models with effective batch
sizes in the millions of tokens get trained on GPUs that can only fit a few thousand tokens at
once: run many small forward/backward passes, accumulate, then step. Mixed precision (bfloat16
compute, float32 master weights) is standard for training anything at LLM scale, since it can
roughly double throughput on modern accelerators. Below, we combine all three ideas into one
small training loop — clipped, accumulated, and run in float16 — and confirm the combination
still trains correctly, matching a full-precision, non-accumulated baseline.

In [ ]:
# 1) Float16 underflow, and the loss-scaling trick that mixed-precision training relies on
tiny_grad = torch.tensor(1e-8, dtype=torch.float32)  # a small but perfectly valid float32 gradient
print(f"Gradient value in float32: {tiny_grad.item():.3e}")
print(f"Same value cast directly to float16: {tiny_grad.half().item():.3e}  <- UNDERFLOWED TO ZERO")

scale = 4096.0
scaled_then_cast = (tiny_grad * scale).half()
recovered = (scaled_then_cast.float() / scale).item()
print(f"\nScale by {scale} BEFORE casting to float16: {scaled_then_cast.item():.3e}")
print(f"Unscale by dividing by {scale} AFTER: {recovered:.3e}")
print(f"Relative error vs. the true float32 value: {abs(recovered - tiny_grad.item())/tiny_grad.item()*100:.4f}%")
print("This scale-before/unscale-after pattern is EXACTLY what GradScaler does automatically.")

# 2) Combine clipping + accumulation + autocast (bfloat16 on CPU) into one realistic training loop
def train_mixed_precision(steps=100, accum_steps=4, micro_bs=8, clip_norm=1.0, lr=0.1):
    model = make_model()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(7)
    losses = []
    for step in range(steps):
        opt.zero_grad()
        for _ in range(accum_steps):
            idx = torch.randint(0, N, (micro_bs,), generator=g)
            with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
                pred = model(X[idx])
                loss = torch.nn.functional.mse_loss(pred, y[idx]) / accum_steps
            loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        opt.step()
        with torch.no_grad():
            losses.append(torch.nn.functional.mse_loss(model(X), y).item())
    return losses

losses_combo = train_mixed_precision()
print(f"\nClipped + accumulated + bfloat16-autocast training: loss {losses_combo[0]:.4f} -> {losses_combo[-1]:.4f}")
print("All three engineering tricks compose cleanly -- none of them changed WHAT is being optimized.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. A model has two parameters with gradients $g_1=(3,4)$ and $g_2=(0,12)$ (as flat vectors).
   Compute the total gradient norm by hand, then compute the clipped gradients for
   `max_norm=5`. Verify with `clip_grad_norm_scratch`. As a sanity check, confirm the clipped
   gradients' own total norm comes out to exactly $5.0$, and that each clipped vector still
   points in the same direction as its unclipped counterpart (i.e. is a positive scalar multiple
   of it).

<details>
<summary>💡 Solution</summary>

$\|g_1\|^2=3^2+4^2=25$, $\|g_2\|^2=0^2+12^2=144$; total norm
$=\sqrt{25+144}=\sqrt{169}=13$. Since $13>5$, clipping applies with coefficient
$c=5/13\approx0.3846$: $g_1\to(1.1538, 1.5385)$, $g_2\to(0, 4.6154)$ — matching
`clip_grad_norm_scratch` exactly. Sanity check: the clipped vectors' combined norm is
$\sqrt{(1.1538^2+1.5385^2)+(0^2+4.6154^2)} = 5.0$ exactly, confirming the clip landed precisely at
the threshold rather than over- or under-shooting; and since both clipped vectors are just $g_1$
and $g_2$ scaled by the same positive constant $c$, each keeps its original direction exactly —
clipping changes *how far* you step, never *which way*.

</details>

### 💭 UNDERSTAND
2. In the Visual Intuition cell, change `outlier_step` to occur at step 100 instead of 30 (well
   into training, once the model has already fit reasonably well), and re-run both the clipped
   and unclipped versions with a much longer horizon (e.g. 300 steps after the outlier) to see
   whether the unclipped run ever actually recovers. Predict first: does giving the unclipped run
   more time change the qualitative outcome?

<details>
<summary>💡 Solution</summary>

No — moving the outlier later doesn't change the qualitative outcome, and giving the unclipped
run hundreds of extra steps afterward doesn't let it recover either. At `outlier_step=30`, the
unclipped run's loss peaks around $4869$ (from a pre-outlier loss of about $0.019$) and, 300
steps later, still sits around $0.28$ — over $14\times$ worse than where it was before the
outlier hit, never getting back within even $2\times$ of its pre-outlier loss. At
`outlier_step=250` (much later in training), the peak is comparably enormous ($\approx3090$) and
the post-outlier recovery is *worse*, not better (final loss around $3.94$, roughly $250\times$
its pre-outlier value) — later in training the model's weights have smaller, more finely-tuned
values, so the same huge corrupted-batch gradient does proportionally more relative damage. The
clipped version, by contrast, barely notices the outlier regardless of timing (peak loss capped
around $0.75$–$0.77$ in both cases, final loss near its pre-outlier baseline). The lesson: one bad
batch isn't a transient hiccup an unclipped optimizer just shrugs off given enough subsequent
steps — without clipping, it can permanently derail a training run's trajectory.

</details>

### ✏️ DERIVE
3. Float16 has a smallest positive representable (subnormal) value of $2^{-24}\approx
   5.96\times10^{-8}$ — any smaller nonzero value rounds to exactly $0$. Derive, as a function of
   a gradient value $g$, the minimum loss-scaling factor $s$ needed so that $g\cdot s$ survives
   casting to float16 without underflowing to zero, and verify your formula against the
   notebook's own example ($g=10^{-8}$, chosen scale $4096$).

<details>
<summary>💡 Solution outline</summary>

Underflow to exactly zero happens whenever the scaled value $g\cdot s$ falls below float16's
smallest positive representable magnitude, $2^{-24}$. To survive, we need
$g\cdot s \ge 2^{-24}$, i.e. $s\ge 2^{-24}/g$. For the notebook's own $g=10^{-8}$: the minimum
scale is $2^{-24}/10^{-8}\approx5.96$ — meaning *any* scale of $6$ or larger would already
prevent underflow, and the notebook's chosen scale of $4096$ clears this minimum with enormous
room to spare (roughly $687\times$ the bare minimum needed). Verified directly: casting
$10^{-8}$ to float16 with no scaling gives exactly $0$ (confirmed underflow); scaling by $4096$
first gives $4.0948\times10^{-5}$ in float16 (representable, since it's far above the smallest
subnormal), and dividing back by $4096$ afterward recovers $9.997\times10^{-9}$ — a relative
error of only about $0.028\%$ from the true value, versus $100\%$ error (total information loss)
with no scaling at all. In practice, `GradScaler` picks a large scale like this dynamically
(growing it when no overflow occurs, shrinking it if one does) rather than requiring you to
derive the bare minimum by hand, but the derivation shows *why* almost any reasonably large
constant works far below the point of causing float16 overflow on the other end.

</details>

### 🐍 IMPLEMENT
4. Real training loops clip the gradient *once*, after accumulation, right before the optimizer
   step — never clipping each micro-batch's gradient individually first. Implement and compare
   both orderings directly: (a) clip each micro-batch's gradient to `max_norm` individually,
   *then* sum the clipped gradients; versus (b) sum the raw micro-batch gradients first, and
   clip the accumulated total exactly once. Use 4 micro-batches where one has an outlier-scale
   gradient, and check whether the two orderings produce the same final (post-clip) gradient.

<details>
<summary>✅ How to know you're right</summary>

The two orderings should produce **different** results whenever any individual micro-batch's own
gradient exceeds `max_norm` on its own — verified directly: with one outlier micro-batch (out of
4, each contributing $g_i/4$ to the sum before any clipping), clipping-then-summing produces a
final accumulated gradient norm of about $1.77$, while summing-then-clipping-once correctly caps
the final norm at exactly `max_norm=1.0` (the raw accumulated norm before that final clip was
about $40.6$). Clipping each micro-batch individually is *wrong* here: it caps each piece's
contribution separately, but nothing then stops their sum from exceeding `max_norm` again once
added together — exactly the gap between $1.77$ and the intended $1.0$. This is precisely why
`torch.nn.utils.clip_grad_norm_` is called once, after `loss.backward()` has been invoked for
every micro-batch in the accumulation window and right before `optimizer.step()` — never inside
the per-micro-batch loop.

</details>

### 🤖 APPLY
5. `04.02`/`04.03`/`04.06` established that a fixed learning rate has a stability ceiling set by
   the loss surface's curvature — too large, and gradient descent diverges outright, regardless
   of outliers. Test whether gradient clipping changes that ceiling: using the notebook's own
   model, data, and (outlier-free) training setup, find the largest learning rate that trains
   stably *without* clipping, then check whether learning rates well above that ceiling become
   stable once clipping is turned on.

<details>
<summary>✅ What you should observe</summary>

Without clipping, this problem's stability ceiling sits at roughly $\eta\approx0.487$ — learning
rates at or below about $0.48$ train to completion, while $\eta=0.5$ and above diverge outright
within the first handful of steps. With `clip_grad_norm=1.0` turned on, learning rates far above
that ceiling ($\eta=1, 5, 10$, even $10\times$ or more above the unclipped ceiling) all remain
numerically stable (no divergence to `inf`/`nan`), though they don't necessarily reach as *good*
a final loss as a well-tuned smaller rate — final loss climbs steadily from about $0.016$ at
$\eta=0.1$ up to double digits by $\eta=5$–$10$. The mechanism is direct: clipping caps how far
any single step can move regardless of what $\eta$ multiplies, so a huge nominal learning rate
simply means "always take a step of length `max_norm`" rather than "take a step whose length
scales unboundedly with the gradient" — clipping doesn't repeal the curvature-based instability
`04.02`/`04.03` describes, it just puts a hard ceiling under it that keeps even a badly-mistuned
learning rate from diverging outright, at the cost of that same ceiling now limiting how fast a
*well*-tuned run can converge too.

</details>

### 🚀 CHALLENGE
6. Gradient accumulation naturally dilutes the effect of one outlier micro-batch, even with no
   clipping at all, simply because its gradient gets averaged in with $k-1$ ordinary
   micro-batches before the optimizer ever sees it. Investigate this dilution quantitatively:
   holding the outlier's magnitude fixed, measure both (a) the norm of the *post-accumulation*
   gradient itself, and (b) the resulting peak in the full-dataset *loss* several steps later, as
   a function of the number of accumulation steps $k\in\{1,2,4,8,16,32\}$ (with `micro_bs` held
   fixed, so a larger $k$ means a larger effective batch). Do both quantities dilute at the same
   rate as $k$ grows, or different rates — and why might they differ?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer finds that the two quantities dilute at genuinely *different* rates, and explains
why. The accumulated gradient's own norm dilutes almost exactly linearly in $1/k$ — a log-log fit
of gradient norm against $k$ gives a slope of about $-1.01$, matching the simple averaging
argument exactly (summing $k-1$ ordinary gradients and one outlier gradient, then dividing by
$k$, shrinks the outlier's *share* of the total roughly proportionally to $1/k$). But the
resulting peak in the *loss* several steps later dilutes noticeably faster — a log-log fit gives
a slope around $-1.9$, closer to quadratic than linear. A strong answer connects this back to
`04.02`/`04.03`'s own optimality-gap analysis: for a locally quadratic loss, the damage from an
oversized step scales with the *squared* norm of the displacement it causes (the loss increase
from a bad step is itself roughly quadratic in how far the step moved you), so a linear reduction
in gradient norm compounds into a roughly quadratic reduction in the resulting loss spike. The
practical takeaway a strong answer draws: larger accumulation windows (larger effective batch
sizes) provide a real, "free" side benefit of built-in outlier robustness that has nothing to do
with clipping at all — though a strong answer also notes this dilution assumes exactly one
corrupted micro-batch per window; the argument would need revisiting if *multiple* micro-batches
in the same accumulation window were simultaneously corrupted.

</details>

---

## 📚 Further Reading
- Pascanu, Mikolov & Bengio, ["On the difficulty of training Recurrent Neural Networks"](https://arxiv.org/abs/1211.5063) (gradient clipping)
- Micikevicius et al., ["Mixed Precision Training"](https://arxiv.org/abs/1710.03740)

---

*Next notebook: [Visualizing Loss Landscapes](09_loss_landscape_visualization.ipynb)*